# Máquinas Térmicas – Lección 16
  
## Plantas térmicas de turbinas de vapor (Ciclo Rankine)

**Autor:** Camilo Bayona  
**Fecha:** 30/10/2025


## Objetivos de aprendizaje

Al finalizar esta lección, el estudiante podrá:
- Reconocer la **arquitectura** de una planta de vapor y la función de cada **componente**.
- Trazar y explicar el **ciclo Rankine** (básico, con **supercalentamiento**, **recalentamiento** y **regeneración**) en diagramas **T–s** y **p–h**.
- Formular **balances de energía** por componente y derivar **rendimientos** (bomba, turbina, caldera y ciclo).
- Discutir el impacto de **presión de caldera**, **vacío en condensador**, **recalentamiento** y **regeneración** sobre la **eficiencia térmica** y la **humedad** en etapas LP.
- Explicar por qué el **tren de intercambiadores de la caldera** (supercalentador, recalentador, evaporador, economizador) sigue un orden específico respecto al flujo de gases de combustión.
- Identificar qué **instrumentación** (termómetros, manómetros, caudalímetros) se necesita para operar y controlar cada una de estas mejoras en una planta real.

In [ ]:
# Instalación solo en Google Colab; en el entorno local del curso es un no-op
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "matplotlib", "ipywidgets", "numpy", "scipy", "CoolProp"], check=True)

In [ ]:
# === Setup =================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Circle, Wedge
from ipywidgets import (interact, interactive_output, FloatSlider, IntSlider,
                        Dropdown, Checkbox, HBox, VBox)
from IPython.display import display
from CoolProp.CoolProp import PropsSI

# Paleta del proyecto (misma familia que Lecciones 7-15; roles nuevos de esta lección)
COL = {"bomba": "#2a78d6", "caldera": "#eb6834", "turbina": "#4a3aa7", "condensador": "#1baf7a",
       "recalentador": "#c0392b", "fwh": "#8a8a86", "economizador": "#1b7a3d",
       "gas": "#b5651d", "agua": "#2a78d6", "vapor": "#8a8a86",
       "perdida": "#e34948", "metal": "#c3c2b7", "tinta": "#3a3a38"}

def plot_dome(fluid, ax, diag="Ts"):
    # Domo de saturación en 'Ts', 'Pv' o 'Ph' (CoolProp) -- misma función de Lecciones 2/15
    Tt = PropsSI("Ttriple", fluid) + 0.5
    Tc_ = PropsSI("Tcrit", fluid) - 0.2
    Ts = np.linspace(Tt, Tc_, 200)
    props = {"liq": {}, "vap": {}}
    for key, Q in (("liq", 0), ("vap", 1)):
        props[key]["s"] = np.array([PropsSI("S", "T", T, "Q", Q, fluid) for T in Ts]) / 1e3
    if diag == "Ts":
        ax.plot(props["liq"]["s"], Ts - 273.15, color="#777777", lw=1.3)
        ax.plot(props["vap"]["s"], Ts - 273.15, color="#777777", lw=1.3)
        ax.set_xlabel("$s$ [kJ/(kg·K)]"); ax.set_ylabel("$T$ [°C]")

def area_ciclo_Ts(s_cycle, T_cycle):
    # Integral de linea cerrada (shoelace) -- area encerrada en T-s, proxy numerico de w_net
    s = np.asarray(s_cycle); T = np.asarray(T_cycle)
    return 0.5*abs(np.sum(s[:-1]*T[1:] - s[1:]*T[:-1]) + (s[-1]*T[0] - s[0]*T[-1]))

print("Setup OK")

##1. Planta térmica de turbina de vapor

Las plantas para la generación de energía basadas en el vapor de agua constan esencialmente de:

1) **Caldera y supercalentadores**  
   Generan **vapor saturado** y elevan su **temperatura** (supercalentamiento) antes de la turbina.

2) **Turbinas de vapor (múltiples cuerpos y extracciones)**  
   HP–MP–LP con **sangrías** para **regeneración** (precalentamiento del agua de alimentación).

3) **Condensador**  
   Licúa el vapor de escape a **baja presión** (vacío), cerrando el ciclo.

4) **Gobernador / control**  
   Regula **caudal** y **entropía** de entrada a turbina (válvulas, control de presión/temperatura).

5) **Calentadores de agua de alimentación**  
   **Cerrados** (superficie) u **abiertos** (contacto directo); recuperan calor y elevan $T$ del agua.

6) **Bombas**  
   Mueven el líquido y elevan su **presión** (p. ej., bomba de condensado y de alimentación).

7) **Generador y auxiliares**  
   Alternador, sistemas eléctricos, **torre de enfriamiento**, **tratamiento de agua**, etc.








In [ ]:
# Diagrama de bloques de la planta (reutilizado en S1 y en el Widget de instrumentacion, S7)
RUTAS = {
    "simple": [("bomba", "Bomba"), ("caldera", "Caldera +\nSobrecalentador"),
               ("turbina", "Turbina"), ("condensador", "Condensador")],
    "recalentamiento": [("bomba", "Bomba"), ("caldera", "Caldera +\nSobrecalentador"),
                         ("turbina", "Turbina HP"), ("recalentador", "Recalentador"),
                         ("turbina", "Turbina MP/LP"), ("condensador", "Condensador")],
    "regeneracion": [("bomba", "Bomba 1"), ("fwh", "FWH\n(abierto)"), ("bomba", "Bomba 2"),
                      ("caldera", "Caldera +\nSobrecalentador"), ("turbina", "Turbina"),
                      ("condensador", "Condensador")],
}

def dibujar_planta_vapor(modo="simple", resaltar=None, instrumentos=False, figsize=(11.5, 3.2)):
    ruta = RUTAS[modo]
    n = len(ruta)
    fig, ax = plt.subplots(figsize=figsize)
    xs = np.linspace(0.9, 0.9 + 1.55*(n-1), n)
    y = 1.0
    for x, (clave, nombre) in zip(xs, ruta):
        fc = COL[clave] if resaltar == clave else "#fdf6e3"
        ec = COL["tinta"] if resaltar != clave else COL[clave]
        lw = 1.1 if resaltar != clave else 2.4
        ax.add_patch(FancyBboxPatch((x - 0.62, y - 0.42), 1.24, 0.84, boxstyle="round,pad=0.05",
                                     fc=fc, ec=ec, lw=lw))
        ax.text(x, y, nombre, ha="center", va="center", fontsize=8.3,
                 fontweight="bold" if resaltar == clave else "normal")
    for k in range(n - 1):
        ax.add_patch(FancyArrowPatch((xs[k] + 0.64, y), (xs[k + 1] - 0.64, y),
                                      arrowstyle="-|>", mutation_scale=12, color=COL["tinta"]))
        if instrumentos:
            xm = (xs[k] + xs[k+1]) / 2
            ax.add_patch(Circle((xm, y + 0.30), 0.11, fc="white", ec=COL["perdida"], lw=1.2, zorder=6))
            ax.text(xm, y + 0.30, "T\nP", ha="center", va="center", fontsize=5.0, zorder=7)
    # lazo de retorno condensador -> primera bomba
    ax.add_patch(FancyArrowPatch((xs[-1], y - 0.45), (xs[0], y - 0.45),
                                  connectionstyle="arc3,rad=-0.3", arrowstyle="-|>",
                                  mutation_scale=12, color=COL["agua"], lw=1.3, ls="--"))
    ax.text((xs[0] + xs[-1]) / 2, y - 1.05, "agua de alimentación (líquido)", fontsize=8,
            ha="center", color=COL["agua"], style="italic")

    if modo == "regeneracion":
        # sangria: de la turbina hacia el FWH (posicion anterior en la ruta), arco POR ENCIMA
        # de las cajas (evita cruzar el lazo de retorno de abajo y las demas cajas)
        i_turb = [k for k, (c, _) in enumerate(ruta) if c == "turbina"][0]
        i_fwh = [k for k, (c, _) in enumerate(ruta) if c == "fwh"][0]
        ax.add_patch(FancyArrowPatch((xs[i_turb], y + 0.44), (xs[i_fwh], y + 0.44),
                                      connectionstyle="arc3,rad=0.35", arrowstyle="-|>",
                                      mutation_scale=11, color=COL["recalentador"], lw=1.3))
        xm = (xs[i_turb] + xs[i_fwh]) / 2
        ax.text(xm, 1.95, "sangría (extracción de vapor)", fontsize=7.8,
                ha="center", color=COL["recalentador"], style="italic")
        if instrumentos:
            ax.add_patch(Circle((xm, 1.55), 0.11, fc="white", ec=COL["economizador"], lw=1.2, zorder=6))
            ax.text(xm, 1.55, "Q", ha="center", va="center", fontsize=6.5, zorder=7, fontweight="bold")

    ax.set_xlim(xs[0]-0.9, xs[-1]+0.9); ax.set_ylim(-1.8, 2.3); ax.axis("off")
    plt.tight_layout(); plt.show()

dibujar_planta_vapor("simple")

##2. Ciclo Rankine standard

El ciclo utilizado para la transformación de energía térmica de vapores es el llamado ciclo de Rankine. Su desarrollo en los últimos cien años ha sido notable, pues a través de una serie de avances técnicos que han resultado en incrementos sucesivos de la temperatura máxima del ciclo y de la capacidad neta de las turbinas y generadores, se ha pasado de sistemas que operaban inicialmente a bajas presiones y bajas temperaturas hasta los sistemas más modernos de los ciclos supercríticos y combinados de hoy en día. La figura 1 muestra un esquema simplificado en el que se observan los cambios de estado del vapor al pasar sucesivamente por la turbina (3-4), el condensador (4-1), la bomba de líquido (1-2) y el generador de calor o caldera (2-3). El proceso ideal que ocurre en la turbina es el de expansión adiabática, es decir, sin evolución de calor, en una sucesión de estados reversibles. Este se asimila a un proceso a entropía constante en el que el vapor saturado a alta presión (punto 3) se expande reversiblemente en la turbina hasta una presión baja (punto 4). El vapor restante es condensado, extrayendo calor QB a temperatura constante TB en el condensador; el líquido se comprime por medio de una bomba hasta la presión de la caldera, en donde se genera vapor en el proceso 2-3 mediante la adición QA.


**Secuencia de estados** (ideal):
- **Bomba:** $1 \rightarrow 2$ (compresión de líquido subenfriado)  
- **Caldera:** $2 \rightarrow 3$ (aporte de calor hasta vapor saturado/sobrecalentado)  
- **Turbina:** $3 \rightarrow 4$ (expansión adiabática ideal, isentrópica)  
- **Condensador:** $4 \rightarrow 1$ (rechazo de calor a presión casi constante)


El widget siguiente resuelve estos 4 estados con CoolProp (agua real) y dibuja el ciclo en el diagrama T–s sobre el domo de saturación — reemplaza la Figura 1 esquemática del original por el ciclo real, a escala, para los parámetros que el estudiante elija.

In [ ]:
# WIDGET 1 -- Ciclo Rankine simple, con el area T-s sombreada = w_net (verificado abajo)
def rama(h_ini, h_fin, P, n=60):
    # tramo a presion constante P, de h_ini a h_fin -- devuelve (s,T) a lo largo del tramo
    hs = np.linspace(h_ini, h_fin, n)
    s = np.array([PropsSI("S", "P", P, "H", h, "Water") for h in hs])
    T = np.array([PropsSI("T", "P", P, "H", h, "Water") for h in hs])
    return s, T

def rama_isentropica(P_ini, P_fin, s, n=60):
    Ps = np.linspace(P_ini, P_fin, n)
    T = np.array([PropsSI("T", "P", p, "S", s, "Water") for p in Ps])
    return np.full(n, s), T

def area_ciclo_Ts(s_loop, T_loop):
    s, T = np.asarray(s_loop), np.asarray(T_loop)
    return 0.5*abs(np.sum(s[:-1]*T[1:] - s[1:]*T[:-1]) + (s[-1]*T[0] - s[0]*T[-1]))

def rankine_simple(P_caldera_kPa=8000.0, P_cond_kPa=10.0, T3_C=500.0):
    P3 = P2 = P_caldera_kPa*1e3
    P1 = P4 = P_cond_kPa*1e3
    h1 = PropsSI("H", "P", P1, "Q", 0, "Water"); s1 = PropsSI("S", "P", P1, "Q", 0, "Water")
    v1 = 1/PropsSI("D", "P", P1, "Q", 0, "Water")
    h2 = h1 + v1*(P2-P1); s2 = PropsSI("S", "P", P2, "H", h2, "Water")
    h3 = PropsSI("H", "P", P3, "T", T3_C+273.15, "Water"); s3 = PropsSI("S", "P", P3, "T", T3_C+273.15, "Water")
    h4 = PropsSI("H", "P", P4, "S", s3, "Water")
    wt, wp, qin = h3-h4, h2-h1, h3-h2
    eta = (wt-wp)/qin

    s12, T12 = rama(h1, h2, P2)                       # bomba (casi vertical en T-s)
    s23, T23 = rama(h2, h3, P2)                        # caldera (isobara)
    s34, T34 = rama_isentropica(P3, P4, s3)             # turbina (isentropica)
    s41 = np.linspace(s3, s1, 60)
    T41 = np.full(60, PropsSI("T", "P", P4, "Q", 0, "Water"))  # condensador (isobara+isoterma)
    s_loop = np.concatenate([s12, s23, s34, s41])
    T_loop = np.concatenate([T12, T23, T34, T41]) - 273.15
    area_kJ = area_ciclo_Ts(s_loop/1e3, T_loop + 273.15)   # kJ/(kg·K) x K = kJ/kg

    fig, ax = plt.subplots(figsize=(7.5, 6))
    plot_dome("Water", ax, "Ts")
    ax.fill(s_loop/1e3, T_loop, color=COL["caldera"], alpha=0.18, zorder=1)
    ax.plot(s_loop/1e3, T_loop, color=COL["caldera"], lw=2.2, zorder=3)
    for k, i in enumerate([(s1, PropsSI("T","P",P1,"Q",0,"Water")-273.15), (s2, T12[-1]-273.15),
                            (s3, T23[-1]-273.15), (s3, T34[-1]-273.15)], start=1):
        ax.annotate(str(k), i, textcoords="offset points", xytext=(6,5), fontsize=10, fontweight="bold")
    caja = (f"$\\eta_{{th}}$ = {eta*100:.1f} %\n$w_{{net}}$ = {(wt-wp)/1e3:.1f} kJ/kg\n"
            f"área T-s = {area_kJ:.1f} kJ/kg\n$q_{{in}}$ = {qin/1e3:.1f} kJ/kg")
    ax.text(0.03, 0.97, caja, transform=ax.transAxes, va="top", fontsize=9.5,
             bbox=dict(boxstyle="round", fc="white", ec=COL["caldera"], alpha=0.92))
    ax.set_title("Ciclo Rankine simple — el área sombreada ES $w_{net}$")
    plt.tight_layout(); plt.show()
    return dict(eta=eta, wnet=wt-wp, area_kJ=area_kJ, qin=qin)

interact(rankine_simple,
         P_caldera_kPa=FloatSlider(8000, min=1000, max=20000, step=500, description="P caldera [kPa]"),
         P_cond_kPa=FloatSlider(10, min=5, max=100, step=5, description="P cond [kPa]"),
         T3_C=FloatSlider(500, min=250, max=600, step=10, description="T₃ [°C]"));

La eficiencia del ciclo teórico se aproxima a la del ciclo de Carnot, que opera entre las temperaturas promedio más altas y más bajas del mismo. Como el rendimiento del ciclo de Carnot viene dado por $1 - TB/TA$, donde $TB$ es la temperatura más baja y $TA$ es la temperatura más alta, es claro que se puede obtener un mejor rendimiento al disminuir la temperatura de extracción de calor $TB$ o al aumentar la temperatura de adición de calor $TA$. En las plantas termoeléctricas, esto se logra mediante la **operación del condensador a una presión por debajo de la atmosférica** o mediante **procesos de supercalentamiento del vapor antes de entrar a la turbina** **o recalentando el vapor una vez que ha sido parcialmente expandido en la turbina**. Estas medidas se complementan en el ciclo actual con esquemas de regeneración, cuyo objetivo es disminuir el calor aportado por el combustible. En la práctica, esto se realiza mediante extracciones de vapor en la turbina, con el fin de precalentar el agua que fluye hacia la caldera.


###2.1 Eficiencia térmica del ciclo

El rendimiento termodinámico del ciclo de Rankine, como ya se ha dicho, depende de las temperaturas promedio de adición de calor en la caldera y de eliminación de calor en el condensador. Por esta razón y en virtud de los consumos de combustibles de las plantas de fuerza que operan en el mundo y que generan la mayor parte de la energía que se consume, se ha tratado de mejorar hasta donde ha sido posible el rendimiento del ciclo. El rendimiento termodinámico interno del ciclo se define como la relación entre el trabajo neto realizado y el calor aportado al vapor. Esta relación se puede definir como:

$$
\eta = \frac{w}{q_H}
$$

Donde se ha considerado, en su determinación, el trabajo y el calor por unidad de masa. El trabajo neto es igual al trabajo de la turbina o el motor menos el trabajo realizado por las bombas de líquido y los equipos auxiliares, es decir:

$$
w = w_{turb} - \Sigma w_{bombas \ y \ auxiliares}
$$

El trabajo de las bombas se puede estimar fácilmente, puesto que es igual a:

$$
w_{bomba \ ideal} = \int v dp = \underline{v} \Delta P
$$

Donde $\underline{v}$ es el volumen específico promedio para el intervalo de presiones considerado. Este sería el trabajo ideal de compresión; sin embargo, las bombas son equipos relativamente ineficientes y el trabajo real es entonces mayor que el dado por esta expresión. La eficiencia de la bomba se define como:

$$
\eta_b = \frac{w_{bomba \ ideal}}{w_{bomba \ actual}}
$$

En este caso, cabe anotar que el rendimiento o eficiencia de las bombas es relativamente bajo, siendo del orden del 50 al 70%.

El calor $q_H$ aportado al vapor es el que gana el agua al pasar por la caldera, más posiblemente el que pueda ganar el vapor al pasar por el supercalentador y los recalentadores. En el caso de la caldera, la energía del combustible no se aprovecha en su totalidad, sino que, como se explica más adelante, una parte va al vapor y el resto se pierde en los gases en la chimenea, por la conducción y radiación térmica a los alrededores y otras pérdidas similares. El rendimiento de la caldera se determina mediante la relación:

$$
\eta_{cal} = \frac{q_{H}}{PCI}
$$

Donde $PCI$ es el poder calorífico inferior del combustible en kJ/kg.


La **eficiencia teórica** se aproxima a la de **Carnot** entre $T_B$ (baja) y $T_A$ (alta):
$$
\eta_{\text{Carnot}} = 1 - \frac{T_B}{T_A}
$$
Mejoras: **disminuir $T_B$** (más vacío en condensador) y **aumentar $T_A$** (mayor $P/T$ de caldera, **supercalentamiento** y **recalentamiento**). La **regeneración** reduce el calor suministrado en caldera al **precalentar** el agua de alimentación con **sangrías** de turbina.


##3. Turbina

En el caso de la turbina o turbinas, dado que es el equipo que realiza la mayor transformación de energía del sistema, su rendimiento interno es muy importante, ya que tiene un gran efecto sobre el rendimiento global. Si el intercambio de energía se considera adiabático, es decir, sin transferencia de calor, entonces, el trabajo ideal por unidad de masa se puede expresar como:

$$
w_i = h_3 - h_{4i}
$$

Donde $h_3$ es la eficiencia isentrópica de la turbina. Obsérvese que en el diagrama $T$ vs $s$, la posición del punto 4 queda a la derecha del punto 3; esto explica un aumento de la entropía sin que se haya realizado transferencia de calor.

En las plantas de turbina de vapor actuales no se permite el uso de vapor húmedo, puesto que las gotitas de agua que contienen erosionan rápidamente las aspas de la turbina. Por esto, el vapor saturado pasa por un intercambiador llamado supercalentador con el fin de que cuando ocurra la expansión, el estado final se encuentre supercalentado o con un contenido mínimo de humedad. Este caso se muestra en la siguiente figura, en la que el punto $4i$ sería el estado final del vapor si la expansión en la turbina ocurriera de forma ideal, mientras que el punto 4 representa un estado más cercano a la realidad de la expansión irreversible en la turbina. Este último se encuentra muy cerca del punto de saturación con un contenido de agua líquida muy bajo.


**Eficiencia isentrópica de turbina** (el texto la menciona pero no la formaliza):

$$\eta_t=\frac{w_{real}}{w_i}=\frac{h_3-h_4}{h_3-h_{4i}}$$

con $h_{4i}$ el estado isentrópico ideal ($s_{4i}=s_3$) y $h_4$ el estado real (algo más a la derecha en T-s, mayor entropía, por la irreversibilidad de la expansión real).

El widget siguiente reemplaza las Figuras 2 y 3 del original: compara, a la **misma $P_{caldera}$**, entrar a la turbina con **vapor saturado** vs. **vapor sobrecalentado**, y cuantifica la calidad $x_4$ a la salida — el motivo cuantitativo (no solo cualitativo) del supercalentamiento.

In [ ]:
# WIDGET 2 -- humedad a la salida: vapor saturado vs. sobrecalentado a la entrada de turbina
def estado_turbina(Pb, Pc, T3_C=None):
    # si T3_C es None, entra vapor SATURADO (Q=1); si no, vapor sobrecalentado a T3_C
    s1 = PropsSI("S", "P", Pc, "Q", 0, "Water")
    if T3_C is None:
        h3 = PropsSI("H", "P", Pb, "Q", 1, "Water"); s3 = PropsSI("S", "P", Pb, "Q", 1, "Water")
        T3 = PropsSI("T", "P", Pb, "Q", 1, "Water")
    else:
        h3 = PropsSI("H", "P", Pb, "T", T3_C+273.15, "Water"); s3 = PropsSI("S", "P", Pb, "T", T3_C+273.15, "Water")
        T3 = T3_C + 273.15
    h4 = PropsSI("H", "P", Pc, "S", s3, "Water")
    try:
        x4 = PropsSI("Q", "P", Pc, "H", h4, "Water")
    except ValueError:
        x4 = -1.0
    return dict(s3=s3, T3=T3, h4=h4, x4=x4)

def humedad_turbina(P_caldera_kPa=8000.0, P_cond_kPa=10.0, T3_sobrecalentado_C=500.0):
    Pb, Pc = P_caldera_kPa*1e3, P_cond_kPa*1e3
    sat = estado_turbina(Pb, Pc, None)
    sob = estado_turbina(Pb, Pc, T3_sobrecalentado_C)

    fig, ax = plt.subplots(figsize=(7.5, 6))
    plot_dome("Water", ax, "Ts")
    for d, color, nombre in [(sat, COL["condensador"], "vapor saturado"), (sob, COL["caldera"], "sobrecalentado")]:
        s34, T34 = rama_isentropica(Pb, Pc, d["s3"], n=80)
        ax.plot(s34/1e3, T34-273.15, color=color, lw=2.3,
                 label=f"{nombre}: $x_4$={d['x4']:.3f}" if 0<=d['x4']<1 else f"{nombre}: seco/sobrecalentado")
        ax.plot(d["s3"]/1e3, d["T3"]-273.15, "o", ms=7, color=color, mec=COL["tinta"], mew=1.2)
    ax.legend(fontsize=9.5, loc="lower right")
    ax.set_title(f"Misma $P_{{caldera}}$={P_caldera_kPa/1e3:.1f} MPa — efecto del sobrecalentamiento en $x_4$")
    plt.tight_layout(); plt.show()
    hum_sat = (1-sat["x4"])*100 if 0<=sat["x4"]<1 else 0.0
    hum_sob = (1-sob["x4"])*100 if 0<=sob["x4"]<1 else 0.0
    print(f"Humedad a la salida -- saturado: {hum_sat:.1f} %   sobrecalentado ({T3_sobrecalentado_C:.0f}°C): "
          f"{hum_sob:.1f} %{'  (seco/sobrecalentado)' if hum_sob==0 else ''}")

interact(humedad_turbina,
         P_caldera_kPa=FloatSlider(8000, min=2000, max=20000, step=500, description="P caldera [kPa]"),
         P_cond_kPa=FloatSlider(10, min=5, max=50, step=5, description="P cond [kPa]"),
         T3_sobrecalentado_C=FloatSlider(500, min=300, max=600, step=10, description="T₃ sobrecal. [°C]"));

##4. Recalentamiento

Una forma posible de mejorar el rendimiento térmico es aumentar la temperatura media de adición de calor. Esto se logra mediante la expansión del vapor de forma parcial en la turbina de alta presión, para ser regresado a la caldera, donde aumenta su temperatura hasta niveles similares a los que tenía inicialmente, para luego ser expandido nuevamente en una turbina de baja presión. Esto podría realizarse muchas veces con el fin de que el ciclo tuviera el aspecto mostrado en la figura 4; sin embargo, las plantas actuales rara vez tienen más de un recalentamiento. El nivel de temperatura al final del recalentamiento (punto 5) puede ser similar al del punto 3. Una ventaja adicional del recalentamiento es que el vapor tiende a salir de la turbina bastante seco, ayudando a proteger el equipo contra la erosión.


Estrategia para **elevar $T$ promedio de adición de calor** y **reducir humedad** en etapas finales:

1. Expansión parcial en **turbina HP**: $3 \rightarrow 4$  
2. **Recalentamiento** en caldera: $4 \rightarrow 5$
3. Expansión en **turbina MP/LP**: $5 \rightarrow 6$

Suele emplearse **un** recalentamiento; $T_5 \approx T_3$.

El widget siguiente reemplaza la Figura 4 (reutiliza y hace interactivos los parámetros de la celda de código del original: $P_{caldera}=15$ MPa, $P_{reheat}=3.5$ MPa, $T_3=T_5=540\,°C$, $\eta_{bomba}=0.85$, $\eta_{t,HP}=0.88$, como valores por defecto de los sliders). Además de $\eta_{th}$ y el área T-s, reporta $x_6$ (calidad a la salida de la turbina LP) — **el original nunca calculaba esta calidad**, así que la ventaja de "sale más seco" quedaba solo en la prosa; aquí se cuantifica.

**Nota honesta sobre el área sombreada.** En el Widget 1 (ciclo ideal, componentes 100% reversibles) el área T-s coincide con $w_{net}$ con error <0.01 % — verificado numéricamente. Aquí, con $\eta_{bomba},\eta_{t,HP}<1$ reales, el área sombreada **excede ligeramente** a $w_{net}$: la turbina HP real no sigue una isentrópica (su entropía de salida $s_4>s_3$), así que el tramo 3→4 se traza como segmento recto entre los dos estados reales — una convención visual estándar, no la trayectoria física exacta — y $\oint T\,ds$ ya no es exactamente $w_{net}$ cuando hay generación de entropía. La diferencia es, de hecho, una medida visual de la **irreversibilidad** (pérdida de trabajo utilizable), no un error de cálculo.

In [ ]:
# WIDGET 3 -- recalentamiento interactivo (misma fisica de la celda del original, con eficiencias reales)
def ciclo_recalentado(Pb, Pr, Pc, T3, T5, eta_p, eta_hp):
    h1 = PropsSI("H","P",Pc,"Q",0,"Water"); s1 = PropsSI("S","P",Pc,"Q",0,"Water")
    h2s = PropsSI("H","P",Pb,"S",s1,"Water")
    wp_ideal = h2s-h1; wp_real = wp_ideal/eta_p; h2 = h1+wp_real
    s2 = PropsSI("S","P",Pb,"H",h2,"Water")
    h3 = PropsSI("H","P",Pb,"T",T3,"Water"); s3 = PropsSI("S","P",Pb,"T",T3,"Water")
    h4s = PropsSI("H","P",Pr,"S",s3,"Water")
    wthp_ideal = h3-h4s; wthp_real = eta_hp*wthp_ideal; h4 = h3-wthp_real
    s4 = PropsSI("S","P",Pr,"H",h4,"Water")
    h5 = PropsSI("H","P",Pr,"T",T5,"Water"); s5 = PropsSI("S","P",Pr,"T",T5,"Water")
    h6s = PropsSI("H","P",Pc,"S",s5,"Water")
    try:
        x6 = PropsSI("Q","P",Pc,"H",h6s,"Water")
    except ValueError:
        x6 = -1.0
    qcal, qreh = h3-h2, h5-h4
    qin = qcal+qreh
    wtlp = h5-h6s
    wnet = wthp_real+wtlp-wp_real
    eta = wnet/qin
    return dict(h=[h1,h2,h3,h4,h5,h6s], s=[s1,s2,s3,s4,s5,s5], P=[Pc,Pb,Pb,Pr,Pr,Pc],
                eta=eta, wnet=wnet, qin=qin, x6=x6)

def recalentamiento(P_caldera_MPa=15.0, P_reheat_MPa=3.5, P_cond_kPa=10.0,
                     T3_C=540.0, T5_C=540.0, eta_bomba=0.85, eta_turb_HP=0.88):
    Pb, Pr, Pc = P_caldera_MPa*1e6, P_reheat_MPa*1e6, P_cond_kPa*1e3
    d = ciclo_recalentado(Pb, Pr, Pc, T3_C+273.15, T5_C+273.15, eta_bomba, eta_turb_HP)
    h1,h2,h3,h4,h5,h6 = d["h"]

    s12,T12 = rama(h1,h2,Pb); s23,T23 = rama(h2,h3,Pb)
    # turbina HP real (eta_hp<1): s4 != s3, el proceso real no sigue ninguna isobara/isentropica
    # simple -- se traza como segmento recto entre los dos ESTADOS REALES (3) y (4), convencion
    # estandar de libro de texto para representar una expansion irreversible en T-s
    T3K = PropsSI("T","P",Pb,"H",h3,"Water"); T4K = PropsSI("T","P",Pr,"H",h4,"Water")
    s34 = np.linspace(d["s"][2], d["s"][3], 60); T34 = np.linspace(T3K, T4K, 60)
    s45,T45 = rama(h4,h5,Pr)
    s56,T56 = rama_isentropica(Pr,Pc,d["s"][4])     # turbina LP: ideal/isentropica (ver nota del widget)
    s61 = np.linspace(d["s"][5], d["s"][0], 60)
    T61 = np.full(60, PropsSI("T","P",Pc,"Q",0,"Water"))
    s_loop = np.concatenate([s12,s23,s34,s45,s56,s61])
    T_loop = np.concatenate([T12,T23,T34,T45,T56,T61])
    area_kJ = area_ciclo_Ts(s_loop/1e3, T_loop)

    fig, ax = plt.subplots(figsize=(7.5,6))
    plot_dome("Water", ax, "Ts")
    ax.fill(s_loop/1e3, T_loop-273.15, color=COL["recalentador"], alpha=0.16, zorder=1)
    ax.plot(s_loop/1e3, T_loop-273.15, color=COL["recalentador"], lw=2.2, zorder=3)
    caja = (f"$\\eta_{{th}}$ = {d['eta']*100:.1f} %\n$w_{{net}}$ = {d['wnet']/1e3:.1f} kJ/kg\n"
            f"área T-s = {area_kJ:.1f} kJ/kg\n$q_{{in}}$ = {d['qin']/1e3:.1f} kJ/kg\n"
            f"$x_6$ = {d['x6']:.3f}" + (" (húmedo)" if 0<=d['x6']<1 else " (seco)"))
    ax.text(0.03, 0.97, caja, transform=ax.transAxes, va="top", fontsize=9.3,
             bbox=dict(boxstyle="round", fc="white", ec=COL["recalentador"], alpha=0.92))
    ax.set_title("Ciclo con recalentamiento — compárese el área y $x_6$ contra el Widget 1")
    plt.tight_layout(); plt.show()

interact(recalentamiento,
         P_caldera_MPa=FloatSlider(15.0, min=5.0, max=25.0, step=0.5, description="P caldera [MPa]"),
         P_reheat_MPa=FloatSlider(3.5, min=1.0, max=8.0, step=0.5, description="P reheat [MPa]"),
         P_cond_kPa=FloatSlider(10.0, min=5.0, max=50.0, step=5.0, description="P cond [kPa]"),
         T3_C=FloatSlider(540.0, min=400.0, max=600.0, step=10.0, description="T₃ [°C]"),
         T5_C=FloatSlider(540.0, min=400.0, max=600.0, step=10.0, description="T₅ (reheat) [°C]"),
         eta_bomba=FloatSlider(0.85, min=0.6, max=1.0, step=0.05, description="η bomba"),
         eta_turb_HP=FloatSlider(0.88, min=0.6, max=1.0, step=0.02, description="η turb. HP"));

##5. Regeneración

Otra manera de mejorar el rendimiento del ciclo de Rankine consiste en incrementar la temperatura del agua que se ha condensado y que se encuentra como líquido subenfriado a la presión del condensador. Por ejemplo, si la presión de condensación fuese de 1 pulgada de mercurio absoluto (1/30 atm), la temperatura del líquido saturado sería relativamente baja (aproximadamente 46 °C). Por otra parte, si la presión de la caldera fuese igual a 4 MPa, la temperatura de saturación sería de unos 190 °C.

Esto quiere decir que se podría calentar el líquido antes de entrar a la caldera a través de un diferencial importante de temperatura. Este calentamiento se podría realizar en una turbina especial en la que el vapor que se expande fluyera en contracorriente con el agua que se está calentando. Teóricamente, se podría obtener agua en estado de saturación a la entrada de la caldera.

De este modo, se disminuiría el aporte de calor en esta última al mismo tiempo que la temperatura de adición de calor del ciclo ideal se haría constante e igual a la temperatura de saturación del vapor a alta presión. Como todo el calor extraído en el condensador fluiría a temperatura constante, la eficiencia térmica del ciclo sería, en teoría, igual a la del ciclo de Carnot.

Tal equipo es prácticamente imposible de construir; no obstante, en la actualidad se puede conseguir un efecto parecido mediante el esquema explicado a continuación.


Eleva la **temperatura del agua de alimentación** antes de caldera, reduciendo $q_H$ y acercando el perfil térmico al **Carnot** ideal.  

Implementación práctica mediante **extracciones** (sangrías) y **calentadores**:

- **Abiertos (contacto directo):** el vapor de sangría se mezcla con el agua → **misma presión**.  
- **Cerrados (superficie):** intercambio sin mezcla → **presiones distintas** posibles.

> Múltiples etapas de calentamiento permiten acercar $T_{\text{agua}}$ a la **temperatura de saturación** de la caldera, reduciendo el **salto térmico** en evaporador y mejorando $\eta$.

**Balance de energía del calentador abierto** (fracción de vapor extraído $y$, por kg de vapor que sale de caldera): el calentador mezcla $y$ kg de vapor de sangría (estado 6) con $(1-y)$ kg de líquido de la bomba 1 (estado 2), entregando líquido saturado (estado 3) a la presión de extracción $P_{extr}$:

$$y=\frac{h_3-h_2}{h_6-h_2}$$

con $q_{in}=h_5-h_4$ — **menor** que el del ciclo simple a la misma $P_{caldera}/T_3$, porque el agua ya entra precalentada. El widget siguiente reemplaza la Figura 5 y compara contra el ciclo simple del Widget 1.

> **Nota sobre el área T-s.** A diferencia de los Widgets 1 y 3 (una sola masa de vapor recorriendo todo el ciclo), aquí el flujo se **divide** en la extracción (estado 6): una fracción $y$ va al calentador, el resto sigue a la turbina. La identidad $\oint T\,ds=w_{net}$ exige una masa fija recorriendo un lazo cerrado, así que no aplica literalmente con una bifurcación de flujo — el área se sombrea solo como referencia visual de forma, no como un segundo cálculo de $w_{net}$.

In [ ]:
# WIDGET 4 -- regeneracion interactiva (1 FWH abierto), comparado contra el ciclo simple
def ciclo_regen(Pb, Pe, Pc, T3):
    h1 = PropsSI("H","P",Pc,"Q",0,"Water"); v1 = 1/PropsSI("D","P",Pc,"Q",0,"Water")
    s1 = PropsSI("S","P",Pc,"Q",0,"Water")
    h2 = h1+v1*(Pe-Pc); s2 = PropsSI("S","P",Pe,"H",h2,"Water")
    h3 = PropsSI("H","P",Pe,"Q",0,"Water"); v3 = 1/PropsSI("D","P",Pe,"Q",0,"Water")
    s3 = PropsSI("S","P",Pe,"Q",0,"Water")
    h4 = h3+v3*(Pb-Pe); s4 = PropsSI("S","P",Pb,"H",h4,"Water")
    h5 = PropsSI("H","P",Pb,"T",T3,"Water"); s5 = PropsSI("S","P",Pb,"T",T3,"Water")
    h6 = PropsSI("H","P",Pe,"S",s5,"Water")
    h7 = PropsSI("H","P",Pc,"S",s5,"Water")
    y = (h3-h2)/(h6-h2)
    wt = (h5-h6)+(1-y)*(h6-h7)
    wp = (1-y)*(h2-h1)+(h4-h3)
    qin = h5-h4
    return dict(h=[h1,h2,h3,h4,h5,h6,h7], s=[s1,s2,s3,s4,s5,s5,s5], P=[Pc,Pe,Pe,Pb,Pb,Pe,Pc],
                eta=(wt-wp)/qin, y=y, wnet=wt-wp, qin=qin)

def regeneracion(P_caldera_kPa=8000.0, P_cond_kPa=10.0, T3_C=500.0, P_extr_kPa=None):
    Pb, Pc, T3 = P_caldera_kPa*1e3, P_cond_kPa*1e3, T3_C+273.15
    Pe = P_extr_kPa*1e3 if P_extr_kPa else np.sqrt(Pb*Pc)
    reg = ciclo_regen(Pb, Pe, Pc, T3)

    # ciclo simple (misma Pb/Pc/T3) para comparar -- recalculado aqui, no via el widget 1
    h1s = PropsSI("H","P",Pc,"Q",0,"Water"); v1s = 1/PropsSI("D","P",Pc,"Q",0,"Water")
    s1s = PropsSI("S","P",Pc,"Q",0,"Water")
    h2s_ = h1s+v1s*(Pb-Pc); s2s = PropsSI("S","P",Pb,"H",h2s_,"Water")
    h3s = PropsSI("H","P",Pb,"T",T3,"Water"); s3s = PropsSI("S","P",Pb,"T",T3,"Water")
    h4s_ = PropsSI("H","P",Pc,"S",s3s,"Water")
    eta_simple = ((h3s-h4s_)-(h2s_-h1s))/(h3s-h2s_)

    fig, ax = plt.subplots(figsize=(7.5,6))
    plot_dome("Water", ax, "Ts")
    Ts_reg = [PropsSI("T","P",Pi,"H",hi,"Water")-273.15 for hi,Pi in zip(reg["h"], reg["P"])]
    s_reg = [si/1e3 for si in reg["s"]]
    ax.fill(s_reg+[s_reg[0]], Ts_reg+[Ts_reg[0]], color=COL["fwh"], alpha=0.18, zorder=1)
    ax.plot(s_reg+[s_reg[0]], Ts_reg+[Ts_reg[0]], "-o", color=COL["fwh"], lw=2.2, ms=5, zorder=3,
             label=f"regeneración (η={reg['eta']*100:.1f}%)")
    Ts_simp = [h1s, h2s_, h3s, h4s_]
    Ps_simp = [Pc, Pb, Pb, Pc]
    s_simp = [s1s/1e3, s1s/1e3, s3s/1e3, s3s/1e3]
    T_simp = [PropsSI("T","P",p,"H",h,"Water")-273.15 for h,p in zip(Ts_simp,Ps_simp)]
    ax.plot(s_simp+[s_simp[0]], T_simp+[T_simp[0]], "--", color=COL["vapor"], lw=1.6, zorder=2,
             label=f"simple, mismas P/T₃ (η={eta_simple*100:.1f}%)")
    ax.legend(fontsize=9, loc="lower right")
    ax.set_title(f"$P_{{extr}}$={Pe/1e3:.0f} kPa   $y$={reg['y']*100:.1f} % extraído")
    plt.tight_layout(); plt.show()
    print(f"q_in regeneración = {reg['qin']/1e3:.1f} kJ/kg   vs.   q_in simple = {(h3s-h2s_)/1e3:.1f} kJ/kg "
          f"({100*(1-reg['qin']/(h3s-h2s_)):.1f} % menos)")

interact(regeneracion,
         P_caldera_kPa=FloatSlider(8000, min=2000, max=20000, step=500, description="P caldera [kPa]"),
         P_cond_kPa=FloatSlider(10, min=5, max=50, step=5, description="P cond [kPa]"),
         T3_C=FloatSlider(500, min=300, max=600, step=10, description="T₃ [°C]"),
         P_extr_kPa=FloatSlider(900, min=200, max=3000, step=100, description="P extracción [kPa]"));

En la práctica, la regeneración se logra mediante la extracción de una parte del vapor durante la expansión en la turbina; esta se utiliza para precalentar el agua de alimentación, haciéndola circular por un intercambiador de calor (regenerador), como se puede ver en la figura 5. El intercambiador puede ser de superficie, es decir, que no hay contacto entre el vapor usado para calentar el líquido que circula por el otro lado del equipo; en este caso, las presiones del vapor extraído y la del líquido pueden ser diferentes. El intercambio térmico también puede llevarse a cabo mediante un intercambiador de contacto directo o abierto, en el que el vapor de extracción entra en contacto con el líquido, elevando su temperatura.

Este tipo de intercambiadores se utiliza, además, para extraer el aire que pueda contener el agua de alimentación y, por lo general, se colocan en la zona de baja presión de la planta. El proceso de calentamiento del agua se puede repetir una o muchas veces, extrayendo vapor de la turbina y utilizándolo para precalentar el agua de alimentación. De esta forma, el vapor intercambia calor en contraflujo con el agua de alimentación hasta llevarla a una temperatura muy cercana a la de saturación correspondiente a la presión de la caldera.


## 6. El tren de intercambiadores de la caldera: por qué van en ese orden

Hasta ahora cada sección trató la caldera como una caja negra que entrega vapor a $T_3$. En la caldera real, el calor **no** llega todo junto: los gases de combustión recorren un solo camino, de la cámara de combustión a la chimenea, cediendo calor sucesivamente al **economizador**, al **evaporador**, al **sobrecalentador** y, si existe, al **recalentador** — y el **orden físico de estos equipos a lo largo del camino de los gases no es arbitrario**.

**La restricción es de segunda ley, no de ingeniería de detalle:** en un intercambiador en contraflujo, el lado que se calienta (agua/vapor) nunca puede superar, en ningún punto, la temperatura del lado que cede calor (gases) en esa misma posición — si lo hiciera, el calor fluiría de frío a caliente. Como los gases se enfrían **monótonamente** a lo largo de un único camino (de la cámara de combustión a la chimenea), el componente que necesita la temperatura **más alta** del lado agua/vapor (el sobrecalentador, que entrega $T_3\approx500$-$600\,°C$) tiene que ubicarse donde los gases están **más calientes** — inmediatamente después de la cámara de combustión. El economizador, que solo necesita calentar agua líquida unos pocos cientos de grados, puede ubicarse donde los gases ya se enfriaron — cerca de la chimenea — y así **recupera calor que de otro modo se perdería**, mejorando $\eta_{cal}$ (§2.1).

> **Nota de alcance.** El widget siguiente es un **perfil cualitativo** (no un diseño de intercambiador con balances de calor/áreas reales) — su objetivo es visualizar la restricción de contraflujo, no dimensionar equipos. La forma de la curva de gases es ilustrativa; la restricción que sí es rigurosa es que la curva de agua/vapor **nunca puede cruzar por encima** de la curva de gases.

In [ ]:
# WIDGET 5 -- perfil de temperaturas en contraflujo a lo largo del tren de la caldera
def perfil_caldera(P_caldera_kPa=8000.0, T3_C=500.0, T_gas_hogar_C=1300.0,
                    T_gas_chimenea_C=180.0, con_recalentador=False):
    Pb = P_caldera_kPa*1e3
    Tsat = PropsSI("T", "P", Pb, "Q", 0, "Water") - 273.15
    T_fw = 180.0   # entrada de agua de alimentacion al economizador (tras FWH/bomba, tipico)

    # zonas a lo largo de x=0 (chimenea, gas frio) -> x=1 (hogar, gas caliente)
    if con_recalentador:
        bordes = [0.0, 0.30, 0.48, 0.72, 0.85, 1.0]
        zonas = ["economizador", "evaporador", "sobrecalentador", "(gas, sin equipo)", "recalentador"]
    else:
        bordes = [0.0, 0.35, 0.55, 0.85]
        zonas = ["economizador", "evaporador", "sobrecalentador"]

    x_gas = np.linspace(0, 1, 200)
    T_gas = T_gas_chimenea_C + (T_gas_hogar_C - T_gas_chimenea_C)*x_gas**0.75

    fig, ax = plt.subplots(figsize=(10.5, 5))
    ax.plot(x_gas, T_gas, color=COL["gas"], lw=2.4, label="gases de combustión")

    colores_zona = {"economizador": COL["economizador"], "evaporador": COL["caldera"],
                     "sobrecalentador": COL["recalentador"], "recalentador": COL["perdida"]}
    x_agua, T_agua = [], []
    xe0, xe1 = bordes[0], bordes[1]
    x_agua += list(np.linspace(xe0, xe1, 40)); T_agua += list(np.linspace(T_fw, Tsat, 40))
    xv0, xv1 = bordes[1], bordes[2]
    x_agua += list(np.linspace(xv0, xv1, 20)); T_agua += [Tsat]*20
    xs0, xs1 = bordes[2], bordes[3]
    x_agua += list(np.linspace(xs0, xs1, 40)); T_agua += list(np.linspace(Tsat, T3_C, 40))
    ax.plot(x_agua, T_agua, color=COL["agua"], lw=2.6, label="agua/vapor principal")
    for (x0, x1), nombre in zip(zip(bordes[:-1], bordes[1:]), zonas):
        if nombre == "(gas, sin equipo)":
            continue
        ax.axvspan(x0, x1, color=colores_zona.get(nombre, "#cccccc"), alpha=0.08)
        ax.text((x0+x1)/2, T_gas_hogar_C*1.03, nombre, ha="center", fontsize=8.5, rotation=0)

    if con_recalentador:
        # la zona "recalentador" ya quedo sombreada y etiquetada por el bucle de zonas de arriba
        xr0, xr1 = bordes[4], bordes[5]
        T4_HP = Tsat + 0.35*(T3_C - Tsat)   # salida turbina HP, ilustrativa (entre Tsat y T3)
        x_rh = np.linspace(xr0, xr1, 40); T_rh = np.linspace(T4_HP, T3_C*0.98, 40)
        ax.plot(x_rh, T_rh, color=COL["perdida"], lw=2.6, label="vapor (recalentador)")

    T_gas_en_agua = np.interp(x_agua, x_gas, T_gas)
    violacion = np.any(np.array(T_agua) > T_gas_en_agua)
    ax.set_xlabel("posición a lo largo del camino de los gases  (chimenea → hogar)")
    ax.set_ylabel("T [°C]")
    titulo = "✘ VIOLACIÓN del contraflujo (agua/vapor más caliente que el gas en algún punto)" if violacion \
             else "✔ perfil físicamente válido (gas siempre más caliente que agua/vapor)"
    ax.set_title(titulo, color=COL["perdida"] if violacion else COL["economizador"], fontsize=10.5)
    ax.legend(fontsize=8.5, loc="center right"); ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()

interact(perfil_caldera,
         P_caldera_kPa=FloatSlider(8000, min=2000, max=20000, step=500, description="P caldera [kPa]"),
         T3_C=FloatSlider(500, min=350, max=600, step=10, description="T₃ [°C]"),
         T_gas_hogar_C=FloatSlider(1300, min=900, max=1600, step=50, description="T gas (hogar) [°C]"),
         T_gas_chimenea_C=FloatSlider(180, min=120, max=300, step=10, description="T gas (chimenea) [°C]"),
         con_recalentador=Checkbox(False, description="incluir recalentador"));

## 7. Instrumentación: cómo se mide esto en una planta real

Todo lo anterior son **modelos**: a partir de $P$, $T$ y una tabla/ecuación de estado (CoolProp en este curso), se calculan $h$, $s$, $x$. En una planta real no se "calculan" los estados — **se miden**, y el modelo solo sirve para interpretar esas medidas y decidir si la planta opera como se espera.

- **Termómetros/termopares (T):** en cada punto numerado del ciclo — son baratos y rápidos, y junto con un manómetro alcanzan para fijar el estado termodinámico completo de una sustancia pura en una sola fase.
- **Manómetros (P):** igual de necesarios en cada punto — $T$ sola no basta (dentro del domo de saturación, $T$ y $P$ no son independientes: todos los puntos del domo a la misma $P$ tienen la misma $T$). Por eso en la zona de dos fases (ej. a la salida de la turbina LP) hace falta un tercer instrumento.
- **Caudalímetros (ṁ):** no aparecen en el ciclo simple (una sola masa recircula sin dividirse), pero se vuelven **imprescindibles** en cuanto el flujo se bifurca: una sangría de regeneración se diseña para una fracción $y$ ideal (§5), pero en operación real esa fracción **se mide y se controla con una válvula**, no se asume — sin caudalímetro no hay forma de saber si el calentador está recibiendo el vapor que el diseño supone.

El widget siguiente muestra, sobre el mismo diagrama de bloques de la §1, dónde se necesita cada instrumento en los 3 casos, y por qué el número crece con la complejidad del ciclo.

In [ ]:
# WIDGET 6 -- instrumentacion necesaria segun la complejidad del ciclo
RESUMEN_INSTRUMENTOS = {
    "simple": "4 pares T/P (estados 1-4). Sin caudalímetros: una sola masa recircula, nada que repartir.",
    "recalentamiento": "6 pares T/P (estados 1-6, incluye entrada/salida del recalentador). "
                        "El par T/P a la salida de caldera y a la salida del recalentador deben "
                        "coincidir en diseño (T₃≈T₅) — una desviación avisa de ensuciamiento o fuga.",
    "regeneracion": "6 pares T/P (estados 1-4 del lazo principal + entrada/salida del FWH) "
                     "**+ 1 caudalímetro en la línea de sangría**: es la única forma de verificar en "
                     "operación que la fracción extraída y real coincide con la y de diseño (§5) — "
                     "el modelo ideal la calcula, la planta real la debe medir y controlar con una válvula.",
}

def widget_instrumentacion(modo="regeneracion"):
    dibujar_planta_vapor(modo, instrumentos=True)
    print(RESUMEN_INSTRUMENTOS[modo])

interact(widget_instrumentacion,
         modo=Dropdown(options=["simple", "recalentamiento", "regeneracion"], value="regeneracion",
                        description="ciclo"));

## Notas de diseño y operación

- **Aumentar $P/T$ de caldera** y **reducir $p_{\text{cond}}$** (vacío) mejora $\eta$, limitado por **materiales** y **termohidráulica**.  
- **Recalentamiento**: disminuye **humedad** en LP y mejora $\eta$ a costa de mayor complejidad.  
- **Regeneración**: disminuye $q_H$ en caldera y mejora el **perfil térmico**.  
- **Tratamiento de agua**: sílice, oxígeno disuelto, pH, **corrosión/ensuciamiento**.  
- **Auxiliares** (bombas/ventiladores): su consumo impacta **$w_{\text{neto}}$**.  
- **Condensador/torre**: control de **vacío**, **fugas de aire**, **ensuciamiento** y **rechazo de calor**.
